<a href="https://colab.research.google.com/github/TnsSamanvitha18/Advanced-ResNet50-Autoencoder-Framework-for-Multi-Class-Eye-Disease-Classification/blob/main/Advanced_ResNet50_Autoencoder_Framework_for_Multi_Class_Eye_Disease_Classification_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Advanced ResNet50-Autoencoder Framework for Multi-Class Eye Disease Classification**

**Binary Classification : Cataract v/s Normal**

Importing Required Libraries

In [ ]:
# Import required libraries for deep learning, dataset loading, ResNet50 transfer learning, and plotting

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.optimizers import Adam

Loading Dataset (Only Cataract, Normal)

In [ ]:
# Load dataset, set training parameters, and create optimized train/validation data pipelines

DATA_DIR = "/content/drive/MyDrive/4 - 2 Semester Project/6th April 2026/Dataset/Cataract_Normal"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 20
train_ds = image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)
val_ds = image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

Residual Autoencoder

In [ ]:
# Build a lightweight residual autoencoder to perform slight denoising before feeding images to ResNet

def build_autoencoder(img_shape):
    inp = layers.Input(shape=img_shape)
    # small noise correction only
    x = layers.Conv2D(16, 3, activation='relu', padding='same')(inp)
    x = layers.Conv2D(16, 3, activation='relu', padding='same')(x)
    correction = layers.Conv2D(3, 3, padding='same')(x)
    # residual connection
    out = layers.Add()([inp, 0.1 * correction])
    autoencoder = models.Model(inp, out)
    autoencoder.compile(optimizer='adam', loss='mse')
    return autoencoder
autoencoder = build_autoencoder((224,224,3))

Training Autoencoder

In [ ]:
# Train autoencoder in an unsupervised manner for denoising, then freeze its weights

print("Training Autoencoder...")
autoencoder.fit(
    train_ds.map(lambda x,y: (x,x)),
    epochs=5,
    validation_data=val_ds.map(lambda x,y: (x,x))
)
autoencoder.trainable = False


Final Model (Autoencoder + ResNet50)

In [ ]:
# Build final classification model: denoised images from autoencoder → ResNet50 → binary classifier

base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)
base_model.trainable = False
inputs = layers.Input(shape=(224,224,3))
# denoised image goes to ResNet
x = autoencoder(inputs)
x = base_model(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu",
                 kernel_regularizer=regularizers.l2(0.01))(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = models.Model(inputs, outputs)
model.compile(
    optimizer=Adam(1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

Train Classifier

In [ ]:
# Train the final model and store training history for accuracy/loss visualization

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

Evaluate Model

In [ ]:
# Evaluate model performance on training and validation sets (Cataract v/s Normal)

train_loss, train_acc = model.evaluate(train_ds)
val_loss, val_acc = model.evaluate(val_ds)
print(f"Training Accuracy (Binary Classification - Cataract v/s Normal): {train_acc*100:.2f}%")
print(f"Validation Accuracy (Binary Classification - Cataract v/s Normal): {val_acc*100:.2f}%")


Plot Training Curves

In [ ]:
# Plot simple training and validation accuracy/loss curves

plt.figure(figsize=(12,5))
# Accuracy
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.title('Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
# Loss
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.title('Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()


Additional Metrics

In [ ]:
# Compute evaluation metrics (Precision, Recall, F1-score) for Glaucoma vs Normal classification

from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import seaborn as sns
# Collect true labels and predictions
y_true = []
y_pred = []
for images, labels in val_ds:
    preds = model.predict(images)
    preds = (preds > 0.5).astype("int32").flatten()
    labels = labels.numpy().astype("int32").flatten()
    y_true.extend(labels)
    y_pred.extend(preds)
# Convert to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)
# Classification report (dictionary format)
report = classification_report(y_true, y_pred, target_names=["Normal", "Cataract"], output_dict=True)
# Convert to DataFrame for table view
df_report = pd.DataFrame(report).transpose()
print("\nClassification Report (Table):\n")
print(df_report)

Confusion Matrix

In [ ]:
# Plot confusion matrix to visualize prediction performance (Cataract v/s Normal)

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Normal", "Cataract"],
    yticklabels=["Normal", "Cataract"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Cataract vs Normal")
plt.show()


**Binary Classification : Diabetic Retinopathy v/s Normal**

Importing Required Libraries

In [ ]:
# Import required libraries for deep learning, dataset loading, ResNet50 transfer learning, and plotting

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.optimizers import Adam

Loading Dataset (Only Diabetic Retinopathy, Normal)

In [ ]:
# Load dataset, set training parameters, and create optimized train/validation data pipelines

DATA_DIR = "/content/drive/MyDrive/4 - 2 Semester Project/6th April 2026/Dataset/DiabeticRetinopathy_Normal"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 20
train_ds = image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)
val_ds = image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)


Residual Autoencoder

In [ ]:
# Build a lightweight residual autoencoder to perform slight denoising before feeding images to ResNet

def build_autoencoder(img_shape):
    inp = layers.Input(shape=img_shape)
    # small noise correction only
    x = layers.Conv2D(16, 3, activation='relu', padding='same')(inp)
    x = layers.Conv2D(16, 3, activation='relu', padding='same')(x)
    correction = layers.Conv2D(3, 3, padding='same')(x)
    # residual connection
    out = layers.Add()([inp, 0.1 * correction])
    autoencoder = models.Model(inp, out)
    autoencoder.compile(optimizer='adam', loss='mse')
    return autoencoder
autoencoder = build_autoencoder((224,224,3))

Training Autoencoder

In [ ]:
# Train autoencoder in an unsupervised manner for denoising, then freeze its weights

print("Training Autoencoder...")
autoencoder.fit(
    train_ds.map(lambda x,y: (x,x)),
    epochs=5,
    validation_data=val_ds.map(lambda x,y: (x,x))
)
autoencoder.trainable = False

Final Model (Autoencoder + ResNet50)

In [ ]:
# Build final classification model: denoised images from autoencoder → ResNet50 → binary classifier

base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)
base_model.trainable = False
inputs = layers.Input(shape=(224,224,3))
# denoised image goes to ResNet
x = autoencoder(inputs)
x = base_model(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu",
                 kernel_regularizer=regularizers.l2(0.01))(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = models.Model(inputs, outputs)
model.compile(
    optimizer=Adam(1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

Train Classifier

In [ ]:
# Train the final model and store training history for accuracy/loss visualization

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

Evaluate Model

In [ ]:
# Evaluate model performance on training and validation sets (Diabetic Retinopathy v/s Normal)

train_loss, train_acc = model.evaluate(train_ds)
val_loss, val_acc = model.evaluate(val_ds)

print(f"Training Accuracy (Binary Classification - Diabetic Retinopathy v/s Normal): {train_acc*100:.2f}%")
print(f"Validation Accuracy (Binary Classification - Diabetic Retinopathy v/s Normal): {val_acc*100:.2f}%")


Plot Training Curves

In [ ]:
# Plot simple training and validation accuracy/loss curves

plt.figure(figsize=(12,5))
# Accuracy
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.title('Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
# Loss
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.title('Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

Additional Metrics

In [ ]:
# Compute evaluation metrics (Precision, Recall, F1-score) for Glaucoma vs Normal classification

from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import seaborn as sns
# Collect true labels and predictions
y_true = []
y_pred = []
for images, labels in val_ds:
    preds = model.predict(images)
    preds = (preds > 0.5).astype("int32").flatten()
    labels = labels.numpy().astype("int32").flatten()
    y_true.extend(labels)
    y_pred.extend(preds)
# Convert to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)
# Classification report (dictionary format)
report = classification_report(y_true, y_pred, target_names=["Normal", "Diabetic Retinopathy"], output_dict=True)
# Convert to DataFrame for table view
df_report = pd.DataFrame(report).transpose()
print("\nClassification Report (Table):\n")
print(df_report)

Confusion Matrix

In [ ]:
# Plot confusion matrix to visualize prediction performance (Diabetic Retinpathy v/s Normal)

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Normal", "Diabetic Retinopathy"],
    yticklabels=["Normal", "Diabetic Retinopathy"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Diabetic Retinopathy vs Normal")
plt.show()

**Binary Classification : Glaucoma v/s Normal**

Importing Required Libraries

In [ ]:
# Import required libraries for deep learning, dataset loading, ResNet50 transfer learning, and plotting

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.optimizers import Adam

Loading Dataset (Only Glaucoma, Normal)

In [ ]:
# Load dataset, set training parameters, and create optimized train/validation data pipelines

DATA_DIR = "/content/drive/MyDrive/4 - 2 Semester Project/6th April 2026/Dataset/Glaucoma_Normal"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 20
train_ds = image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)
val_ds = image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

Residual Autoencoder

In [ ]:
# Build a lightweight residual autoencoder to perform slight denoising before feeding images to ResNet

def build_autoencoder(img_shape):
    inp = layers.Input(shape=img_shape)
    # small noise correction only
    x = layers.Conv2D(16, 3, activation='relu', padding='same')(inp)
    x = layers.Conv2D(16, 3, activation='relu', padding='same')(x)
    correction = layers.Conv2D(3, 3, padding='same')(x)
    # residual connection
    out = layers.Add()([inp, 0.1 * correction])
    autoencoder = models.Model(inp, out)
    autoencoder.compile(optimizer='adam', loss='mse')
    return autoencoder
autoencoder = build_autoencoder((224,224,3))

Training Autoencoder

In [ ]:
# Train autoencoder in an unsupervised manner for denoising, then freeze its weights

print("Training Autoencoder...")
autoencoder.fit(
    train_ds.map(lambda x,y: (x,x)),
    epochs=5,
    validation_data=val_ds.map(lambda x,y: (x,x))
)
autoencoder.trainable = False

Final Model (Autoencoder + ResNet50)

In [ ]:
# Build final classification model: denoised images from autoencoder → ResNet50 → binary classifier

base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)
base_model.trainable = False
inputs = layers.Input(shape=(224,224,3))
# denoised image goes to ResNet
x = autoencoder(inputs)
x = base_model(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu",
                 kernel_regularizer=regularizers.l2(0.01))(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = models.Model(inputs, outputs)
model.compile(
    optimizer=Adam(1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

Train Classifier

In [ ]:
# Train the final model and store training history for accuracy/loss visualization

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

Evaluate Model

In [ ]:
# Evaluate model performance on training and validation sets (Glaucoma v/s Normal)

train_loss, train_acc = model.evaluate(train_ds)
val_loss, val_acc = model.evaluate(val_ds)
print(f"Training Accuracy (Binary Classification - Glaucoma v/s Normal): {train_acc*100:.2f}%")
print(f"Validation Accuracy (Binary Classification - Glaucoma v/s Normal): {val_acc*100:.2f}%")


Plot Training Curves

In [ ]:
# Plot simple training and validation accuracy/loss curves

plt.figure(figsize=(12,5))

# Accuracy
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.title('Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
# Loss
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.title('Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

Additional Metrics

In [ ]:
# Compute evaluation metrics (Precision, Recall, F1-score) for Glaucoma vs Normal classification

from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import seaborn as sns
# Collect true labels and predictions
y_true = []
y_pred = []
for images, labels in val_ds:
    preds = model.predict(images)
    preds = (preds > 0.5).astype("int32").flatten()
    labels = labels.numpy().astype("int32").flatten()
    y_true.extend(labels)
    y_pred.extend(preds)
# Convert to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)
# Classification report (dictionary format)
report = classification_report(y_true, y_pred, target_names=["Normal", "Glaucoma"], output_dict=True)
# Convert to DataFrame for table view
df_report = pd.DataFrame(report).transpose()
print("\nClassification Report (Table):\n")
print(df_report)

Confusion Matrix

In [ ]:
# Plot confusion matrix to visualize prediction performance (Glaucoma v/s Normal)

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Normal", "Glaucoma"],
    yticklabels=["Normal", "Glaucoma"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Glaucoma vs Normal")
plt.show()

**Multi - Class Classification : Cataract v/s Diabetic Retinopathy v/s Glaucoma v/s Normal**

Importing Required Libraries

In [ ]:
# Import required libraries for deep learning, dataset loading, ResNet50 transfer learning, and plotting

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.optimizers import Adam

Loading Dataset (Cataract, Diabetic Retinopathy, Glaucoma, Normal)

In [ ]:
# Load multi-class dataset (Cataract, Diabetic Retinopathy, Glaucoma, Normal) and prepare train/validation pipelines

DATA_DIR = "/content/drive/MyDrive/4 - 2 Semester Project/6th April 2026/Dataset/C_D_G_N"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 25
NUM_CLASSES = 4
train_ds = image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical"
)
val_ds = image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical"
)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)


Residual Autoencoder

In [ ]:
# Build a lightweight residual autoencoder to slightly denoise images before passing them to ResNet

def build_autoencoder(img_shape):
    inp = layers.Input(shape=img_shape)
    x = layers.Conv2D(16, 3, activation='relu', padding='same')(inp)
    x = layers.Conv2D(16, 3, activation='relu', padding='same')(x)
    correction = layers.Conv2D(3, 3, padding='same')(x)
    # residual connection (very small modification)
    out = layers.Add()([inp, 0.1 * correction])
    autoencoder = models.Model(inp, out)
    autoencoder.compile(optimizer='adam', loss='mse')
    return autoencoder
autoencoder = build_autoencoder((224,224,3))


Training Autoencoder

In [ ]:
# Train the autoencoder for image reconstruction (unsupervised) and freeze it before classification

print("Training Autoencoder...")
autoencoder.fit(
    train_ds.map(lambda x,y: (x,x)),
    epochs=5,
    validation_data=val_ds.map(lambda x,y: (x,x))
)
autoencoder.trainable = False

Final Model (Autoencoder + ResNet50)

In [ ]:
# Build final multi-class model: autoencoder-denoised images → ResNet50 → softmax classifier

base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)
base_model.trainable = False
inputs = layers.Input(shape=(224,224,3))
# Autoencoder output → ResNet
x = autoencoder(inputs)
x = base_model(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation="relu",
                 kernel_regularizer=regularizers.l2(0.01))(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
model = models.Model(inputs, outputs)
model.compile(
    optimizer=Adam(1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

Train Classifier

In [ ]:
# Train the multi-class model and store history for accuracy and loss visualization

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

Evaluate Model

In [ ]:
# Evaluate model performance for multi-class eye disease classification (Cataract, DR, Glaucoma, Normal)

train_loss, train_acc = model.evaluate(train_ds)
val_loss, val_acc = model.evaluate(val_ds)
print(f"Training Accuracy (Multi-Class - Catract v/s Diabetic Retinopathy v/s Glaucoma v/s Normal): {train_acc*100:.2f}%")
print(f"Validation Accuracy (Multi-Class - Catract v/s Diabetic Retinopathy v/s Glaucoma v/s Normal): {val_acc*100:.2f}%")


Plot Training Curves

In [ ]:
# Plot training and validation accuracy/loss curves for multi-class classification

plt.figure(figsize=(12,5))
# Accuracy
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.title('Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
# Loss
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.title('Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

Additional Metrics

In [ ]:
# Generate Precision, Recall, and F1-score report for multi-class eye disease classification

from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
y_true = []
y_pred = []
for images, labels in val_ds:
    preds = model.predict(images)
    preds = np.argmax(preds, axis=1)
    labels = np.argmax(labels.numpy(), axis=1)
    y_true.extend(labels)
    y_pred.extend(preds)
y_true = np.array(y_true)
y_pred = np.array(y_pred)
class_names = ["Cataract", "Diabetic Retinopathy", "Glaucoma", "Normal"]
print("\nClassification Report (Multi-Class - C/D/G/N):\n")
print(classification_report(y_true, y_pred, target_names=class_names))

Confusion Matrix

In [ ]:
# Plot confusion matrix to visualize prediction performance for multi-class eye disease classification

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7,6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Multi-Class Classification")
plt.show()


In [ ]:
model.save("eye_disease_model.h5")
model.save_weights("eye_model.weights.h5")

**Inference Code**

In [ ]:
!pip install ipywidgets

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.optimizers import Adam

# 🔧 Rebuild autoencoder EXACTLY SAME
def build_autoencoder(img_shape):
    inp = layers.Input(shape=img_shape)

    x = layers.Conv2D(16, 3, activation='relu', padding='same')(inp)
    x = layers.Conv2D(16, 3, activation='relu', padding='same')(x)

    correction = layers.Conv2D(3, 3, padding='same')(x)

    # ⚠️ keep same (no lambda needed here)
    out = layers.Add()([inp, 0.1 * correction])

    autoencoder = models.Model(inp, out)
    autoencoder.trainable = False

    return autoencoder

autoencoder = build_autoencoder((224,224,3))

# ResNet
base_model = ResNet50(weights="imagenet", include_top=False, input_shape=(224,224,3))
base_model.trainable = False

# Final model
inputs = layers.Input(shape=(224,224,3))
x = autoencoder(inputs)
x = base_model(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(0.01))(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(4, activation="softmax")(x)

model = models.Model(inputs, outputs)

# Compile (needed for loading weights properly)
model.compile(optimizer=Adam(1e-4), loss="categorical_crossentropy", metrics=["accuracy"])

In [ ]:
model.load_weights("eye_model.weights.h5")
print("Model loaded successfully using weights!")

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import io

# Class names (same as training)
class_names = ["Cataract", "Diabetic Retinopathy", "Glaucoma", "Normal"]

# Upload widget
upload = widgets.FileUpload(
    accept='image/*',
    multiple=False
)
display(upload)

def predict_uploaded_image(change):
    if upload.value:
        clear_output(wait=True)
        display(upload)

        for filename, file_info in upload.value.items():
            content = file_info['content']

            # Load image
            img = Image.open(io.BytesIO(content)).convert("RGB")
            img = img.resize((224, 224))

            # ⚠️ IMPORTANT: same preprocessing as training
            img_array = np.array(img)
            img_array = np.expand_dims(img_array, axis=0)

            # Predict
            preds = model.predict(img_array)

            # Show image
            plt.imshow(img)
            plt.axis('off')
            plt.title("Uploaded Image")
            plt.show()

            # Get prediction
            pred_class = np.argmax(preds)
            confidence = np.max(preds)

            print(f"Prediction: {class_names[pred_class]}")
            print(f"Confidence: {confidence:.4f}")

# Trigger prediction
upload.observe(predict_uploaded_image, names='value')